# Child Utils
Parent-driven child table builder.
Call via `%run ../child_utils`

Supports:
- Array explode (attendees, to, contacts, emails, etc.)
- Struct unpivot (envelope directions)
Reuses write_silver, get_silver_watermark, and CDC delete+append from utils.

In [0]:
# Parent-driven child table builder. Call via %run ../child_utils

from pyspark.sql.functions import col, explode_outer, lit, row_number
from pyspark.sql.window import Window

def process_child_table(
    parent_table,
    child_table,
    array_col=None,
    struct_col=None,
    directions=None,
    carry_cols=None,
    parent_key=None,          # None → call find_pk
    dropna_subset=None,
    extra_struct_fields=None
):
    """
    Build a child table from an existing silver parent.
    """
    # Watermark on the child
    try:
        watermark = get_silver_watermark(child_table)
    except Exception:
        watermark = None

    # Read parent
    src = spark.table(f"crm_ingestion.silver.{parent_table}")

    if watermark is not None:
        wm_str = watermark.strftime("%Y-%m-%d %H:%M:%S.%f")
        print(f"CDC MODE: parents updated after {wm_str}")
        src = src.filter(col("bronze_updated_at") > watermark)
    else:
        print("FULL LOAD: all parent rows, latest version per key")
        wm_str = None

    # Detect parent key if not supplied
    if parent_key is None:
        parent_key = find_pk(src)
        print(f"  Detected parent_key = {parent_key}")

    # Latest version of parent per key
    version_window = Window.partitionBy(parent_key).orderBy(
        col("bronze_updated_at").desc(),
        col("bronze_insert_date").desc()
    )
    df_in = (
        src.withColumn("_v", row_number().over(version_window))
           .filter(col("_v") == 1)
           .drop("_v")
    )

    if watermark is not None and df_in.limit(1).count() == 0:
        print("  No changed parents — nothing to do")
        return

    # Build carry_cols if not supplied (now that df_in exists)
    if carry_cols is None:
        scalar_cols = [
            f.name for f in df_in.schema.fields
            if not isinstance(f.dataType, (ArrayType, StructType))
            and f.name != array_col
            and f.name != struct_col
        ]
        carry_cols = list(dict.fromkeys(
            ["bronze_insert_date", "bronze_updated_at", parent_key] + scalar_cols
        ))

    # CDC delete of affected child rows
    if watermark is not None:
        spark.sql(f"""
            DELETE FROM crm_ingestion.silver.{child_table}
            WHERE activity_id IN (
                SELECT {parent_key}
                FROM crm_ingestion.silver.{parent_table}
                WHERE bronze_updated_at > '{wm_str}'
            )
        """)
        global _CDC_WATERMARK
        _CDC_WATERMARK = watermark

    # ---------- Build the child DataFrame ----------
    if array_col:
        select_exprs = [col(c) for c in carry_cols if c in df_in.columns]
        select_exprs.append(explode_outer(col(array_col)).alias("_element"))
        df_child = df_in.select(*select_exprs)

        if "_element" in df_child.columns:
            element_type = df_child.schema["_element"].dataType
            if hasattr(element_type, "fields"):
                expanded = [col(f"_element.`{f.name}`").alias(f.name) for f in element_type.fields]
                df_child = df_child.select(
                    *[c for c in df_child.columns if c != "_element"],
                    *expanded
                )
            else:
                df_child = df_child.withColumnRenamed("_element", array_col)

    elif struct_col and directions:
        pieces = []
        for d in directions:
            piece = df_in.select(
                *[col(c) for c in carry_cols if c in df_in.columns],
                lit(d).alias("direction"),
                explode_outer(col(f"{struct_col}.{d}")).alias("_e"),
                *[col(f"{struct_col}.{f}").alias(f"envelope_{f}") for f in (extra_struct_fields or [])]
            )
            if "_e" in piece.columns:
                e_type = piece.schema["_e"].dataType
                if hasattr(e_type, "fields"):
                    expanded = [col(f"_e.`{f.name}`").alias(f.name) for f in e_type.fields]
                    piece = piece.select(
                        *[c for c in piece.columns if c != "_e"],
                        *expanded
                    )
            pieces.append(piece)

        df_child = pieces[0]
        for p in pieces[1:]:
            df_child = df_child.unionByName(p, allowMissingColumns=True)

    else:
        raise ValueError("Provide either array_col or (struct_col + directions)")
    # ... array or struct explode code that builds df_child ...

    # Always detect the child's own PK
    child_pk = find_pk(df_child)
    print(f"  Child PK = {child_pk}")

    if dropna_subset:
        df_child = df_child.dropna(subset=dropna_subset, how="all")

    write_silver(df_child, child_table)

    if watermark is not None:
        _CDC_WATERMARK = None

    print(f"✅ {child_table} written")